# Discord Persona Inference: Mistral NeMo 12B

This notebook is dedicated exclusively to running multi-turn chat inference with your fine-tuned LoRA adapter. It is designed to be lightweight and state-aware, meaning it will skip redundant downloads if you need to restart the Colab runtime.

### Features:
* **Discord Message Mimicry:** Automatically splits multi-line outputs and adds a typing delay to simulate rapid-fire Discord messaging.
* **Automated Benchmarking:** Paste custom prompts dynamically to test and save the outputs to Google Drive. Use `/benchmark` to run and `/benchmark --reset` to clear them.
* **Dynamic Adapter Switching:** Hot-swap personas mid-conversation without reloading the 12B base model using the `/switch` command.
* **Dynamic Settings:** Change hyperparameters like temperature and repetition penalty on the fly without reloading the model.
* **Multi-Turn Memory:** The clone remembers the conversation history for realistic context using a sliding window to prevent memory crashes.
* **User Awareness:** Formats text to explicitly tell the model who it is talking to.
* **Encrypted Adapter Support:** Automatically downloads and decrypts your adapter from Google Drive.
* **SDPA Attention:** Uses PyTorch's native Scaled Dot-Product Attention for faster generation speeds.

In [ ]:
# ==========================================
# USER CONFIGURATION PARAMETERS
# ==========================================

# 1. Adapter (Clone) Source Configuration
GDRIVE_ADAPTER_LINK = ''
DECRYPTION_KEY = ""

# 2. Base Model Configuration
HF_TOKEN = ""

# 3. Benchmark Configuration
BENCHMARK_OUTPUT_PATH = '/content/drive/MyDrive/soulclone/soulclone_benchmarks.txt'


In [ ]:
import os
import time
import datetime
from google.colab import drive

drive.mount('/content/drive')

def log_print(message):
    current_time = datetime.datetime.now().strftime("%H:%M:%S")
    print(f"[{current_time}] {message}")

log_print("Environment initialized.")

In [ ]:
log_print("Installing dependencies...")
!pip install -q transformers peft bitsandbytes gdown accelerate pyzipper huggingface_hub
log_print("Dependencies installed.")

In [ ]:
import gdown
import pyzipper
import shutil

local_adapter_zip = '/content/adapter.zip'
local_adapter_dir = '/content/adapter_local'

def get_gdrive_id(url):
    if '/d/' in url:
        return url.split('/d/')[1].split('/')[0]
    elif 'id=' in url:
        return url.split('id=')[1].split('&')[0]
    return url

log_print("Checking for existing adapter...")
if os.path.exists(local_adapter_dir) and os.path.isdir(local_adapter_dir) and len(os.listdir(local_adapter_dir)) > 0:
    log_print(f"Adapter already extracted at: {local_adapter_dir}. Skipping download.")
else:
    if GDRIVE_ADAPTER_LINK:
        log_print("Downloading adapter zip from Google Drive via gdown...")
        file_id = get_gdrive_id(GDRIVE_ADAPTER_LINK)
        download_url = f'https://drive.google.com/uc?id={file_id}'
        gdown.download(download_url, local_adapter_zip, quiet=False)
        
        if os.path.exists(local_adapter_zip):
            os.makedirs(local_adapter_dir, exist_ok=True)
            log_print("Extracting adapter...")
            try:
                if DECRYPTION_KEY:
                    log_print("Using DECRYPTION_KEY for AES-256 decryption...")
                    with pyzipper.AESZipFile(local_adapter_zip, 'r', compression=pyzipper.ZIP_DEFLATED, encryption=pyzipper.WZ_AES) as zip_ref:
                        zip_ref.pwd = DECRYPTION_KEY.encode('utf-8')
                        zip_ref.extractall(local_adapter_dir)
                else:
                    with pyzipper.AESZipFile(local_adapter_zip, 'r') as zip_ref:
                        zip_ref.extractall(local_adapter_dir)
                log_print("Adapter successfully extracted.")
                os.remove(local_adapter_zip)
            except Exception as e:
                log_print(f"Extraction failed (Check password): {e}")
    else:
        log_print("WARNING: No GDRIVE_ADAPTER_LINK provided.")

In [ ]:
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from huggingface_hub import login

log_print("Executing aggressive memory clearing...")
for var_name in ['model', 'base_model', 'tokenizer']:
    if var_name in globals():
        del globals()[var_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

if HF_TOKEN:
    log_print("Authenticating with Hugging Face...")
    login(HF_TOKEN)
else:
    log_print("Warning: No HF_TOKEN provided. Download will likely fail.")

base_model_id = "mistralai/Mistral-Nemo-Instruct-2407"

log_print("Configuring 4-bit quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

log_print("Loading Tokenizer (Slow mode)... ")
tokenizer = AutoTokenizer.from_pretrained(base_model_id, use_fast=False)
tokenizer.pad_token = tokenizer.eos_token

log_print("Loading Base Model (Will use cached files if runtime was merely restarted)...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
    attn_implementation="sdpa"
)
base_model.config.torch_dtype = torch.float16
base_model.config.use_cache = True

log_print("Merging LoRA Adapter...")

actual_adapter_path = local_adapter_dir
if not os.path.exists(os.path.join(actual_adapter_path, 'adapter_config.json')):
    nested_dir = os.path.join(actual_adapter_path, 'final_adapter')
    if os.path.exists(os.path.join(nested_dir, 'adapter_config.json')):
        actual_adapter_path = nested_dir

try:
    model = PeftModel.from_pretrained(base_model, actual_adapter_path)
    log_print("Model successfully loaded and merged.")
except Exception as e:
    log_print(f"Failed to merge adapter. Ensure the adapter directory contains the weights: {e}")

In [ ]:
# ******************************************
# Inference Hyperparameters & Chat Interface
# ******************************************

import os
import gdown
import pyzipper
import shutil
import datetime
import time

SYSTEM_PROMPT = "You are lustsoul in a Discord chat."
MAX_NEW_TOKENS = 150
TEMPERATURE = 0.85
TOP_P = 0.9
REPETITION_PENALTY = 1.15
MAX_HISTORY_MESSAGES = 10

conversation_history = [
    {"role": "system", "content": SYSTEM_PROMPT}
]
dynamic_benchmarks = []

if 'active_adapter_name' not in globals():
    active_adapter_name = "default"
if 'adapter_counter' not in globals():
    adapter_counter = 0

def load_new_adapter(gdrive_link, password=""):
    global active_adapter_name, adapter_counter
    
    print(f"\n[Downloading new adapter from Drive...]")
    adapter_counter += 1
    new_adapter_name = f"adapter_{adapter_counter}"
    zip_path = f"/content/temp_adapter_{adapter_counter}.zip"
    extract_dir = f"/content/temp_adapter_{adapter_counter}_extracted"
    
    def get_gdrive_id(url):
        if '/d/' in url: return url.split('/d/')[1].split('/')[0]
        elif 'id=' in url: return url.split('id=')[1].split('&')[0]
        return url
        
    file_id = get_gdrive_id(gdrive_link)
    download_url = f'https://drive.google.com/uc?id={file_id}'
    
    try:
        gdown.download(download_url, zip_path, quiet=True)
        if not os.path.exists(zip_path):
            return "[Error: Failed to download from link.]"
            
        print("[Extracting adapter...]")
        os.makedirs(extract_dir, exist_ok=True)
        if password:
            with pyzipper.AESZipFile(zip_path, 'r', compression=pyzipper.ZIP_DEFLATED, encryption=pyzipper.WZ_AES) as zip_ref:
                zip_ref.pwd = password.encode('utf-8')
                zip_ref.extractall(extract_dir)
        else:
            with pyzipper.AESZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(extract_dir)
        
        actual_path = extract_dir
        if not os.path.exists(os.path.join(actual_path, 'adapter_config.json')):
            nested = os.path.join(actual_path, 'final_adapter')
            if os.path.exists(os.path.join(nested, 'adapter_config.json')):
                actual_path = nested
        
        if not os.path.exists(os.path.join(actual_path, 'adapter_config.json')):
            return "[Error: adapter_config.json not found in the extracted files.]"
            
        print("[Hot-swapping weights into GPU...]")
        model.load_adapter(actual_path, adapter_name=new_adapter_name)
        model.set_adapter(new_adapter_name)
        
        if active_adapter_name:
            model.delete_adapter(active_adapter_name)
            
        active_adapter_name = new_adapter_name
        
        if os.path.exists(zip_path):
            os.remove(zip_path)
            
        return "[Adapter successfully switched! The clone now has a new brain.]"
    except Exception as e:
        return f"[Error switching adapter: {e}]"

def chat_with_clone(username, user_message):
    global conversation_history
    
    formatted_message = f"[{username}]: {user_message}"
    conversation_history.append({"role": "user", "content": formatted_message})
    
    if len(conversation_history) > MAX_HISTORY_MESSAGES + 1:
        trimmed_history = [conversation_history[0]] + conversation_history[-MAX_HISTORY_MESSAGES:]
        if trimmed_history[1]['role'] == 'assistant':
            trimmed_history.pop(1)
        conversation_history = trimmed_history
    
    prompt = tokenizer.apply_chat_template(conversation_history, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        **inputs, 
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        repetition_penalty=REPETITION_PENALTY,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    conversation_history.append({"role": "assistant", "content": response})
    
    return response

def collect_benchmarks():
    global dynamic_benchmarks
    print("\n========================================")
    print("          BENCHMARK SETUP\n")
    print("Please paste your custom benchmarks and press Enter.")
    print("========================================\n")
    
    raw_lines = []
    while True:
        try:
            line = input("> ")
        except EOFError:
            break
            
        if not line.strip() or line.strip().upper() == 'END':
            break
            
        raw_lines.append(line)
        
        if '\n' in line or '\r' in line:
            break
            
    full_text = " ".join(raw_lines).replace("END", "").strip()
    dynamic_benchmarks = []
    
    if '|' in full_text:
        chunks = full_text.split('|')
        current_user = chunks[0].strip()
        
        for i in range(1, len(chunks)):
            chunk = chunks[i].strip()
            
            if i == len(chunks) - 1:
                if current_user and chunk:
                    dynamic_benchmarks.append((current_user, chunk))
            else:
                parts = chunk.rsplit(maxsplit=1)
                if len(parts) == 2:
                    msg = parts[0].strip()
                    next_user = parts[1].strip()
                    if current_user and msg:
                        dynamic_benchmarks.append((current_user, msg))
                    current_user = next_user
                elif len(parts) == 1:
                    current_user = parts[0].strip()
                    
    print(f"\n[Successfully parsed {len(dynamic_benchmarks)} benchmarks:]")
    for user, msg in dynamic_benchmarks:
        print(f"  - [{user}]: {msg}")
    print()

def run_benchmark():
    global active_adapter_name, conversation_history, dynamic_benchmarks
    
    if not dynamic_benchmarks:
        collect_benchmarks()
        if not dynamic_benchmarks:
            print("[No valid benchmarks provided. Aborting benchmark run.]")
            return
            
    print(f"\n[Running {len(dynamic_benchmarks)} benchmarks for adapter '{active_adapter_name}'...]")
    results = f"\n========================================\n"
    results += f"BENCHMARK RUN | Adapter: {active_adapter_name} | {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
    results += f"Settings: Temp={TEMPERATURE}, Pen={REPETITION_PENALTY}\n"
    results += f"========================================\n\n"
    
    for i, (user, msg) in enumerate(dynamic_benchmarks):
        conversation_history = [{"role": "system", "content": SYSTEM_PROMPT}]
        print(f"  Testing prompt {i+1}/{len(dynamic_benchmarks)}: [{user}]...")
        
        reply = chat_with_clone(user, msg)
        
        print(f"    [{user}]: {msg}")
        # Print multiline benchmark responses properly without sleep
        for line in reply.split('\n'):
            if line.strip():
                print(f"    Clone: {line.strip()}")
        print()
        
        results += f"[{user}]: {msg}\nClone: {reply}\n\n"
        
    try:
        os.makedirs(os.path.dirname(BENCHMARK_OUTPUT_PATH), exist_ok=True)
        with open(BENCHMARK_OUTPUT_PATH, "a", encoding="utf-8") as f:
            f.write(results)
        print(f"[Benchmarks complete! Saved to {BENCHMARK_OUTPUT_PATH}]")
    except Exception as e:
        print(f"[Error saving benchmark file: {e}]")
        
    conversation_history = [{"role": "system", "content": SYSTEM_PROMPT}]

print("\n" + "*"*65)
print("Clone is ready! Commands:")
print(" 'quit'  - Exit the chat")
print(" 'reset' - Clear memory and change user")
print(" '/benchmark' - Run prompts and save outputs to Drive")
print(" '/benchmark --reset' - Wipe stored custom benchmarks")
print(" '/temp 0.8' - Change temperature on the fly")
print(" '/pen 1.15' - Change repetition penalty on the fly")
print(" '/switch <link> [password]' - Hot-swap to a new persona adapter")
print("*"*65 + "\n")

current_user = input("Enter your username to begin: ")
print(f"\n[Now chatting as {current_user}]\n")

while True:
    user_input = input(f"{current_user}: ")
    
    if user_input.lower() in ['quit', 'exit', 'stop']:
        print("Ending chat.")
        break
    elif user_input.lower() == 'reset':
        conversation_history = [{"role": "system", "content": SYSTEM_PROMPT}]
        print("[Memory cleared.]")
        current_user = input("Enter new username to begin: ")
        print(f"\n[Now chatting as {current_user}]\n")
        continue
    elif user_input.lower().startswith('/benchmark'):
        if '--reset' in user_input.lower():
            dynamic_benchmarks = []
            print("[Custom benchmarks wiped. You will be prompted to enter new ones next time.]")
        else:
            run_benchmark()
        continue
    elif user_input.lower().startswith('/temp '):
        try:
            TEMPERATURE = float(user_input.split()[1])
            print(f"[Temperature updated to {TEMPERATURE}]")
        except:
            print("[Invalid. Example: /temp 0.85]")
        continue
    elif user_input.lower().startswith('/pen '):
        try:
            REPETITION_PENALTY = float(user_input.split()[1])
            print(f"[Repetition Penalty updated to {REPETITION_PENALTY}]")
        except:
            print("[Invalid. Example: /pen 1.15]")
        continue
    elif user_input.lower().startswith('/switch '):
        parts = user_input.split()
        if len(parts) < 2:
            print("[Invalid. Example: /switch <gdrive_link> [password]]")
            continue
        
        new_link = parts[1]
        new_pw = parts[2] if len(parts) > 2 else ""
        
        status = load_new_adapter(new_link, new_pw)
        print(status)
        
        conversation_history = [{"role": "system", "content": SYSTEM_PROMPT}]
        print("[Memory cleared for new persona.]")
        continue
        
    reply = chat_with_clone(current_user, user_input)
    
    # FIX: Simulate Discord back-to-back messaging behavior by splitting newlines
    for line in reply.split('\n'):
        if line.strip():
            print(f"Clone: {line.strip()}")
            time.sleep(1.5)  # Mimics typing delay for each subsequent message
    print()
